# coerce-float-arg-to-array — worked example 2: mul_grad: coerce the scalar operand, then unbroadcast the grad

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A backward function for `multiply(x, c)` where `c` is a Python `float` cannot compute `grad * c` reliably until `c` is an array — and the gradient w.r.t. `x` must be **unbroadcast** back to `x`'s shape if `c` was a scalar that broadcast against a vector. Coercing the scalar to a 0-D array first makes the shape bookkeeping uniform: a 0-D array broadcasts identically to the Python literal, but is now a real ndarray the unbroadcast logic can inspect.

## Worked solution

**Step 1 — coerce the operand.** We wrap `c` with `np.array(c)` only if it isn't already an `np.ndarray`. After this, `c` is a 0-D array, so `grad * c` is plain array math instead of array-times-Python-float (which works, but loses the uniform-ndarray invariant the wrapper relies on).

**Step 2 — local derivative.** For `out = x * c`, `d out / d x = c`. So the unreduced gradient flowing into `x` is `grad_out * c`, broadcast to the shape of `out`.

**Step 3 — unbroadcast.** Because `c` was a scalar, `out` has `x`'s shape here, so no reduction is needed for `x`; but we still run the general unbroadcast routine so the function is correct even when `c` is a 0-D array that broadcast trivially. Unbroadcast sums over any axes that were expanded and reshapes to the target shape.

**Step 4 — verify against torch autograd.** We build the same computation with `requires_grad=True`, call `.backward()`, and compare. The numbers match because coercion did not change the math — it only made the types uniform.

**Why it works:** coercing first means the unbroadcast helper always receives ndarrays, so `np.shape(...)` and axis-summation never hit a bare Python `float`.

In [ ]:
import numpy as np

def unbroadcast(grad, target_shape):
    # Sum out leading axes that broadcasting added, then sum axes that were size 1.
    while grad.ndim > len(target_shape):
        grad = grad.sum(axis=0)
    for ax, size in enumerate(target_shape):
        if size == 1:
            grad = grad.sum(axis=ax, keepdims=True)
    return grad.reshape(target_shape)

def mul_grad_x(grad_out, x, c):
    if not isinstance(c, np.ndarray):
        c = np.array(c)          # coerce scalar -> 0-D ndarray
    grad_x = grad_out * c        # local derivative d(x*c)/dx = c
    return unbroadcast(grad_x, x.shape)

np.random.seed(0)
x = np.random.randn(4)
c = 3.0
grad_out = np.ones(4)
gx = mul_grad_x(grad_out, x, c)

# Independent check via torch autograd.
xt = t.tensor(x, requires_grad=True)
(xt * c).sum().backward()
print("grad_x        :", np.round(gx, 4))
print("torch grad_x  :", np.round(xt.grad.numpy(), 4))
print("match:", np.allclose(gx, xt.grad.numpy()))